# NHMM for crypto regime detection

Non-homogeneous HMM : transitions depend on external covariates. Useful when you have features (volatility, funding rate, macro indicators) that drive regime changes.

**This notebook** : simulate BTC-like returns with vol-driven regime switching, fit an NHMM, inspect A_t variability across time.


## 1. Simulate regime-switching returns

Two regimes (bull / bear), transition probability depends on realized volatility.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
T = 1500

# Synthetic realized vol — slow-moving covariate
realized_vol = rng.gamma(2, 0.5, T).cumsum() / np.arange(1, T + 1)
realized_vol = (realized_vol - realized_vol.mean()) / realized_vol.std()

# Generate regime + observations
regime = np.zeros(T, dtype=int)
for t in range(1, T):
    p_stay = 0.95 if realized_vol[t] < 0 else 0.7
    regime[t] = regime[t - 1] if rng.random() < p_stay else 1 - regime[t - 1]

X = np.array([rng.normal(0.5 if r == 0 else -0.8, 1.0) for r in regime]).reshape(-1, 1)
Z = realized_vol.reshape(-1, 1)

print(f"T = {T}, regime 0 share = {(regime == 0).mean():.2%}")

## 2. Declare topology and fit NHMM

In [ ]:
from hmm_core.topology import Topology, EmissionSpec, FitSpec, InitSpec
from hmm_core.nhmm import fit_nhmm

topo = Topology(
    name="crypto_2regime_nhmm",
    n_states=2,
    state_names=["bull", "bear"],
    emission=EmissionSpec(type="gaussian", covariance_type="diag", n_features=1),
    allowed_transitions=None,
    startprob="uniform",
    init=InitSpec(strategy="kmeans", seed=42),
    fit=FitSpec(algorithm="baum_welch", n_iter=100, tol=1e-4),
)

result = fit_nhmm(topo, X, Z, covariate_names=["realized_vol"], seed=42)
result

## 3. Inspect time-varying transition matrix

The HTML display above shows :
- **A_t averaged over T** : the homogeneous-equivalent transition matrix
- **A_t variability across t (std)** : where the covariate creates the most variation

Now let's look at specific timesteps.

In [ ]:
# Pick low-vol and high-vol timesteps
low_vol_t = int(np.argmin(Z[:, 0]))
high_vol_t = int(np.argmax(Z[:, 0]))

A_low = result.A_at(low_vol_t)
A_high = result.A_at(high_vol_t)

print(f"At t={low_vol_t} (low vol, z={Z[low_vol_t, 0]:.2f}) :")
print(A_low)
print()
print(f"At t={high_vol_t} (high vol, z={Z[high_vol_t, 0]:.2f}) :")
print(A_high)

## 4. Decode the regime path

In [ ]:
decoded = result.base.model.predict(X)
accuracy = (decoded == regime).mean()
swap_accuracy = (decoded == 1 - regime).mean()
print(f"Decoded vs true regime accuracy : {max(accuracy, swap_accuracy):.2%}")

## Next : GMM-NHMM for multi-modal regimes

If each regime has internal sub-modes (e.g. bull-smooth vs bull-explosive), use `fit_gmm_nhmm` to model them with GMM emissions per state.

If you have multiple independent regime dimensions (trend × volatility × macro), use `fit_factorial_nhmm` to model them as parallel chains.